# Cloud Storage Demand Forecasting, Cost Optimization, and Policy Simulation
## Using Time-Series Machine Learning Models

**Author:** Jagannath Panigrahi  
**Registration No:** 2407102003  
**Supervisor:** Dr. Suvendra Kumar Jayasingh, Associate Professor & HOD  
**Institution:** Institute of Management and Information Technology (IMIT), Cuttack  
**Degree:** M. Tech in Computer Science & Engineering (Data Science)  
**University:** Biju Patnaik University of Technology (BPUT)  

---

### About This Notebook

This notebook is the **complete, reproducible implementation** of the research published in the thesis.  
It walks through every step — from data generation to cost optimization — in a clean, readable manner  
so that fellow researchers can understand, verify, and extend the work.

**Pipeline overview:**
1. Install dependencies  
2. Import libraries  
3. Generate synthetic cloud storage workloads  
4. Visualise and explore the data  
5. Train and evaluate forecasting models (ARIMA, Holt-Winters, XGBoost)  
6. Compare models across metrics and forecasting horizons  
7. Apply a dynamic cost optimization policy  
8. Cross-workload stability analysis  
9. Master results summary  

> **Reproducibility note:** A fixed random seed (`np.random.seed(42)`) is set at the start  
> so every run produces identical results.

---


---
## Step 1 — Install Required Libraries

Run this cell once to install all Python packages needed for the notebook.  
If you are on Google Colab or a fresh environment, this handles everything automatically.

| Package | Purpose |
|---|---|
| `numpy` | Numerical arrays and random number generation |
| `pandas` | Dataset construction and time-series indexing |
| `matplotlib` | Plotting all charts and visualisations |
| `scikit-learn` | Evaluation metrics (RMSE, MAE) and data utilities |
| `statsmodels` | ARIMA and Holt-Winters model implementations |
| `xgboost` | Gradient-boosted decision tree regressor |


In [ ]:
# Install all required packages
# Run this cell once — restart the runtime if prompted
!pip install numpy pandas matplotlib scikit-learn statsmodels xgboost --quiet


---
## Step 2 — Import Libraries

We import everything upfront so dependencies are visible in one place.  
This is standard practice for readable, shareable notebooks.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Evaluation metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Classical statistical forecasting models
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Machine learning forecasting model
from xgboost import XGBRegressor

print("All libraries imported successfully.")


---
## Step 3 — Set Random Seed for Reproducibility

Setting `np.random.seed(42)` guarantees that every run of this notebook  
produces **exactly the same dataset, the same spike events, and the same results**.  
This is a critical requirement for open, reproducible research.


In [ ]:
# Fix the random seed — must be set before any random number generation
np.random.seed(42)

print("Random seed set to 42. Results will be identical on every run.")


---
## Step 4 — Define the Time Axis

We simulate **one full calendar year (365 days)** of cloud storage demand,  
starting from 1 January 2025.

- `time` is an integer array `[0, 1, 2, ..., 364]` used in mathematical formulas.  
- `dates` is a proper `DatetimeIndex` used for readable axis labels in plots.


In [ ]:
# Total simulation period: one calendar year
days = 365
time = np.arange(days)   # Integer time index [0, 1, ..., 364]

# Human-readable date range for plotting
dates = pd.date_range(start="2025-01-01", periods=days)

print(f"Simulation period : {dates[0].date()} to {dates[-1].date()}")
print(f"Total observations: {days} daily data points")


---
## Step 5 — Define Base Storage Parameters

All four workloads share the same starting point and linear growth rate,  
so that differences between them are purely driven by seasonal, bursty,  
or mixed effects — not by different baselines.

| Parameter | Value | Meaning |
|---|---|---|
| `base_storage` | 100 GB | Storage capacity at day 0 |
| `growth_rate` | 0.05 GB/day | Linear organic growth (≈18 GB over the year) |


In [ ]:
base_storage = 100    # Starting storage in GB
growth_rate  = 0.05   # GB added per day (linear trend)

print(f"Base storage  : {base_storage} GB")
print(f"Growth rate   : {growth_rate} GB/day")
print(f"Expected end  : {base_storage + growth_rate * (days - 1):.1f} GB by day {days}")


---
## Step 6 — Generate the Four Synthetic Workload Profiles

We generate four workloads, each representing a realistic cloud storage scenario.  
The mathematical model for each is:

$$y_t = S_0 + g \cdot t + c \cdot \sin\left(\frac{2\pi t}{P}\right) + \varepsilon_t + B_t$$

| Symbol | Meaning |
|---|---|
| $S_0 = 100$ | Base storage (GB) |
| $g = 0.05$ | Daily growth rate |
| $c = 10$ | Seasonal amplitude (GB) |
| $P = 7$ | Seasonal period (weekly cycle) |
| $\varepsilon_t \sim \mathcal{N}(0, 4)$ | Gaussian noise |
| $B_t$ | Random spike event (Bursty/Mixed only) |

### Workload Descriptions

| Workload | Pattern | Real-world analogy |
|---|---|---|
| **Steady** | Linear growth + noise only | Database backup storage growing with user base |
| **Seasonal** | Linear growth + weekly sine cycle + noise | SaaS product with weekday/weekend usage rhythm |
| **Bursty** | Linear growth + 10 random spike events | Batch-processing cluster or event-driven system |
| **Mixed** | Combines all: trend + seasonality + 8 spikes | Enterprise cloud with heterogeneous demand drivers |


In [ ]:
# ── Workload 1: STEADY ────────────────────────────────────────────────────
# Pure linear growth with Gaussian noise. No seasonality, no spikes.
noise1     = np.random.normal(0, 2, days)
workload1  = base_storage + growth_rate * time + noise1

# ── Workload 2: SEASONAL ──────────────────────────────────────────────────
# Linear growth + weekly sine oscillation (period = 7 days) + noise.
noise2          = np.random.normal(0, 2, days)
seasonal_pattern = 10 * np.sin(2 * np.pi * time / 7)
workload2       = base_storage + growth_rate * time + seasonal_pattern + noise2

# ── Workload 3: BURSTY ────────────────────────────────────────────────────
# Linear growth + noise + 10 random spike events of 20-50 GB each.
noise3      = np.random.normal(0, 2, days)
workload3   = base_storage + growth_rate * time + noise3
spike_days3 = np.random.choice(days, 10, replace=False)
for d in spike_days3:
    workload3[d] += np.random.uniform(20, 50)

# ── Workload 4: MIXED ─────────────────────────────────────────────────────
# Combines trend + seasonality + noise + 8 random spike events (20-40 GB).
noise4     = np.random.normal(0, 2, days)
seasonal4  = 10 * np.sin(2 * np.pi * time / 7)
workload4  = base_storage + growth_rate * time + seasonal4 + noise4
spike_days4 = np.random.choice(days, 8, replace=False)
for d in spike_days4:
    workload4[d] += np.random.uniform(20, 40)

print("All four workloads generated successfully.")
print(f"  Steady   — mean: {workload1.mean():.2f} GB,  std: {workload1.std():.2f} GB")
print(f"  Seasonal — mean: {workload2.mean():.2f} GB,  std: {workload2.std():.2f} GB")
print(f"  Bursty   — mean: {workload3.mean():.2f} GB,  std: {workload3.std():.2f} GB")
print(f"  Mixed    — mean: {workload4.mean():.2f} GB,  std: {workload4.std():.2f} GB")


---
## Step 7 — Combine into a Single DataFrame

We package all four workloads together with the date index  
into one tidy DataFrame. This makes slicing, plotting, and  
cross-workload comparisons straightforward.


In [ ]:
# Combine all workloads into one structured DataFrame
df = pd.DataFrame({
    "Date"    : dates,
    "Steady"  : workload1,
    "Seasonal": workload2,
    "Bursty"  : workload3,
    "Mixed"   : workload4
})

print(f"Dataset shape: {df.shape}  ({df.shape[0]} days × {df.shape[1]} columns)")
print()
df.head(10)   # Preview first 10 rows


---
## Step 8 — Descriptive Statistics

`df.describe()` gives a quick statistical summary of each workload:  
minimum, maximum, mean, standard deviation, and quartiles.

**Expected observations:**
- Steady has the **lowest standard deviation** — minimal variation beyond noise.  
- Bursty and Mixed have **high maximums** due to spike events.  
- Seasonal and Mixed have **slightly higher std** due to the sine oscillation.


In [ ]:
# Full statistical summary of all workloads
print("=== Descriptive Statistics (Storage in GB) ===")
print()
print(df.describe().round(2))


In [ ]:
# Column-wise mean — quick sanity check on expected growth
print("=== Column Means (GB) ===")
print(df.mean(numeric_only=True).round(2))


---
## Step 9 — Visualise All Workloads

### 9a — All Four Workloads on One Chart

Plotting all workloads together shows their relative scale and character.  
Even without statistics, the visual differences are striking:  
Steady is smooth, Seasonal ripples, Bursty spikes, Mixed does all three.


In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df["Date"], df["Steady"],   label="Steady",   linewidth=1.2)
plt.plot(df["Date"], df["Seasonal"], label="Seasonal", linewidth=1.2)
plt.plot(df["Date"], df["Bursty"],   label="Bursty",   linewidth=1.2)
plt.plot(df["Date"], df["Mixed"],    label="Mixed",    linewidth=1.2)
plt.title("All Four Cloud Storage Workload Patterns (2025)", fontsize=14, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Storage Demand (GB)")
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("graph_1.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved as graph_1.png")


### 9b — Individual Workload Plots

Viewing each workload separately makes its structure much clearer,  
especially the weekly rhythm in Seasonal and the spike events in Bursty/Mixed.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
workload_names = ["Steady", "Seasonal", "Bursty", "Mixed"]
colours = ["steelblue", "darkorange", "forestgreen", "crimson"]

for ax, name, colour in zip(axes.flatten(), workload_names, colours):
    ax.plot(df["Date"], df[name], color=colour, linewidth=1.0)
    ax.set_title(f"{name} Workload", fontsize=12, fontweight="bold")
    ax.set_xlabel("Date")
    ax.set_ylabel("Storage (GB)")
    ax.grid(True, alpha=0.3)

plt.suptitle("Individual Cloud Storage Workload Profiles", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("graph_2.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved as graph_2.png")


### 9c — Rolling Statistics on Mixed Workload

Rolling mean and rolling standard deviation are computed over a 7-day window  
on the Mixed workload to expose local trend and volatility clusters.

- **Rolling Mean**: Smooths out noise to reveal the underlying trend.  
- **Rolling Std**: Highlights periods of high volatility (e.g., spike windows).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 7-day rolling mean
df["Mixed"].rolling(window=7).mean().plot(
    ax=axes[0], color="steelblue", linewidth=1.5
)
axes[0].set_title("7-Day Rolling Mean — Mixed Workload", fontweight="bold")
axes[0].set_xlabel("Date"); axes[0].set_ylabel("Storage (GB)")
axes[0].grid(True, alpha=0.3)

# 7-day rolling standard deviation
df["Mixed"].rolling(window=7).std().plot(
    ax=axes[1], color="crimson", linewidth=1.5
)
axes[1].set_title("7-Day Rolling Std Dev — Mixed Workload", fontweight="bold")
axes[1].set_xlabel("Date"); axes[1].set_ylabel("Std Dev (GB)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("graph_3_4.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figures saved as graph_3_4.png")


---
## Step 10 — Train / Test Split (80 / 20)

We focus on the **Mixed workload** for model training and evaluation  
because it is the most complex and closest to real enterprise cloud demand.

An **80/20 chronological split** is applied:
- **Training set**: days 1–292 (1 Jan → 19 Oct 2025) — used to fit models  
- **Test set**: days 293–365 (20 Oct → 31 Dec 2025) — used to evaluate forecasts  

> ⚠️ **Important:** No shuffling is applied. Temporal order must be preserved  
> in time-series forecasting — the future must never leak into training.


In [ ]:
# Select the Mixed workload as the target series
series = df["Mixed"]

# 80/20 chronological split
train_size = int(len(series) * 0.8)
train = series[:train_size]
test  = series[train_size:]

print(f"Total observations : {len(series)}")
print(f"Training set size  : {len(train)} days  ({train.index[0].date()} → {train.index[-1].date()})")
print(f"Test set size      : {len(test)}  days  ({test.index[0].date()} → {test.index[-1].date()})")
print(f"Split ratio        : {len(train)/len(series)*100:.0f}% train / {len(test)/len(series)*100:.0f}% test")


---
## Step 11 — Naive Baseline Models

Two naive baselines set the lower performance bound.  
Any reasonable forecasting model should beat both of these easily.

| Baseline | Strategy | When it fails |
|---|---|---|
| **Naive** | Repeat yesterday's value for every future day | Any trend or seasonality |
| **Moving Average** | Use the 7-day trailing mean from end of training | Bursty spikes |


In [ ]:
# ── Naive Forecast ────────────────────────────────────────────────────────
# Simply propagates the last observed training value forward.
naive_pred = np.repeat(train.iloc[-1], len(test))

# ── Moving Average Forecast ───────────────────────────────────────────────
# Uses the 7-day trailing mean from the end of the training window.
ma_value   = train.rolling(window=7).mean().iloc[-1]
ma_pred    = np.repeat(ma_value, len(test))

print(f"Naive forecast value         : {naive_pred[0]:.2f} GB  (constant for all {len(test)} test days)")
print(f"7-day moving average value   : {ma_pred[0]:.2f} GB  (constant for all {len(test)} test days)")


---
## Step 12 — ARIMA(2, 1, 2) Model

**ARIMA** (AutoRegressive Integrated Moving Average) is one of the most  
widely used classical statistical models for time-series forecasting.

### How it works
- **AR(2)**: Uses the past 2 values to predict the next one  
- **I(1)**: Applies 1st-order differencing to remove the linear trend and achieve stationarity  
- **MA(2)**: Corrects predictions using the past 2 forecast errors  

### Why order (2, 1, 2)?
The orders were chosen by:
1. **ADF test** confirming one round of differencing (`d=1`) makes the Mixed workload stationary  
2. **ACF/PACF plots** showing significant lag-2 autocorrelation patterns  
3. **AIC grid search** confirming ARIMA(2,1,2) has the lowest information criterion score  

> ARIMA works best on near-linear, stationary workloads (i.e., the Steady profile).


In [ ]:
# Fit ARIMA(2, 1, 2) on the training data
print("Fitting ARIMA(2, 1, 2) — this may take a few seconds...")
arima_model = ARIMA(train, order=(2, 1, 2))
arima_fit   = arima_model.fit()

# Generate forecasts for the entire test window
arima_pred  = arima_fit.forecast(steps=len(test))

print(f"ARIMA training complete.")
print(f"Forecast range: {arima_pred.min():.2f} GB — {arima_pred.max():.2f} GB")
print()
# Show a compact model summary (optional — comment out if too verbose)
print(arima_fit.summary().tables[0])


---
## Step 13 — Holt-Winters Exponential Smoothing

**Holt-Winters** extends simple exponential smoothing by separately tracking:
- **Level** ($L_t$): the current average value, weighted by $\alpha$  
- **Trend** ($T_t$): the direction and speed of growth, weighted by $\beta$  

### Configuration used
- `trend="add"` — additive trend, suitable for linear growth (storage grows at ~0.05 GB/day)  
- `seasonal=None` — seasonal component omitted because the 73-day test window is too short  
  for reliable seasonal period estimation  
- $\alpha$ and $\beta$ are **automatically optimised** by the L-BFGS-B algorithm minimising SSE  

> Holt-Winters excels on **Seasonal workloads** where a clear periodic pattern exists.


In [ ]:
# Fit Holt-Winters with additive trend (no seasonal component)
print("Fitting Holt-Winters Exponential Smoothing...")
hw_model = ExponentialSmoothing(
    train,
    trend="add",       # Additive linear trend
    seasonal=None      # No seasonal component for this evaluation window
)
hw_fit   = hw_model.fit(optimized=True)   # Auto-optimise alpha and beta
hw_pred  = hw_fit.forecast(len(test))

print("Holt-Winters training complete.")
print(f"Smoothing level (alpha): {hw_fit.params['smoothing_level']:.4f}")
print(f"Smoothing trend (beta) : {hw_fit.params['smoothing_trend']:.4f}")
print(f"Forecast range         : {hw_pred.min():.2f} GB — {hw_pred.max():.2f} GB")


---
## Step 14 — XGBoost Gradient Boosted Regressor

**XGBoost** treats time-series forecasting as a standard supervised regression task.  
Instead of modelling autocorrelation structure explicitly, we provide the **integer  
time index** as the feature, and XGBoost learns a flexible non-linear trend function.

### How it works
1. Training: `X_train = [0, 1, 2, ..., 291]`, `y_train = train storage values`  
2. The model fits 100 decision trees sequentially, each correcting the residuals of the previous  
3. Forecasting: `X_test = [292, 293, ..., 364]` — the model extrapolates the learned trend  

### Why XGBoost?
- Can capture **non-linear demand dynamics** (spikes, curve changes)  
- Built-in regularisation prevents overfitting  
- Fast inference — sub-millisecond per prediction  
- Handles Bursty and Mixed workloads significantly better than classical models  

> Hyperparameters used: `n_estimators=100`, `learning_rate=0.1`, `max_depth=6`  
> (selected by 5-fold time-series cross-validation GridSearch — see Appendix A)


In [ ]:
# Prepare feature and target arrays
X_train = np.arange(len(train)).reshape(-1, 1)   # Shape: (292, 1)
y_train = train.values                             # Shape: (292,)

# Fit XGBoost regressor
print("Fitting XGBoost regressor...")
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    verbosity=0        # Suppress training logs
)
xgb_model.fit(X_train, y_train)

# Generate predictions for the test window
X_test   = np.arange(len(train), len(train) + len(test)).reshape(-1, 1)
xgb_pred = xgb_model.predict(X_test)

print("XGBoost training complete.")
print(f"Trees built          : {xgb_model.n_estimators}")
print(f"Forecast range       : {xgb_pred.min():.2f} GB — {xgb_pred.max():.2f} GB")


---
## Step 15 — Define Evaluation Metrics

We evaluate all models on three standard forecasting metrics:

| Metric | Formula | Interpretation |
|---|---|---|
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | Penalises large errors quadratically; unit = GB |
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Average absolute error; robust to outliers; unit = GB |
| **sMAPE** | $\frac{100}{n}\sum\frac{2|y_i - \hat{y}_i|}{|y_i| + |\hat{y}_i|}$ | Scale-free % error; bounded [0%, 200%]; symmetric |

sMAPE is particularly useful here because it lets us compare accuracy  
across workloads with different mean levels (e.g., Steady vs. Bursty).


In [ ]:
def smape(actual, pred):
    """
    Symmetric Mean Absolute Percentage Error (sMAPE).
    
    Parameters
    ----------
    actual : array-like  Ground-truth values
    pred   : array-like  Predicted values
    
    Returns
    -------
    float  sMAPE as a percentage [0, 200]
    """
    actual = np.array(actual)
    pred   = np.array(pred)
    return np.mean(2 * np.abs(pred - actual) / (np.abs(actual) + np.abs(pred))) * 100

print("sMAPE function defined and ready.")
print()
# Quick sanity check: perfect forecast should return 0%
print(f"Sanity check — perfect forecast: sMAPE = {smape([100, 200], [100, 200]):.1f}%  (expected 0%)")
print(f"Sanity check — 10 GB off on 100: sMAPE = {smape([100, 100], [110, 90]):.1f}%")


---
## Step 16 — Overall Model Performance on Mixed Workload

We now compute RMSE, MAE, and sMAPE for all five models on the 73-day test set  
and display results in a ranked table.

The two naive baselines (Naive, Moving Average) set the performance floor —  
every trained model must beat them to be considered useful.


In [ ]:
# Compute all metrics for every model
results = pd.DataFrame({
    "Model": ["Naive", "Moving Average", "ARIMA", "Holt-Winters", "XGBoost"],
    "RMSE": [
        np.sqrt(mean_squared_error(test, naive_pred)),
        np.sqrt(mean_squared_error(test, ma_pred)),
        np.sqrt(mean_squared_error(test, arima_pred)),
        np.sqrt(mean_squared_error(test, hw_pred)),
        np.sqrt(mean_squared_error(test, xgb_pred)),
    ],
    "MAE": [
        mean_absolute_error(test, naive_pred),
        mean_absolute_error(test, ma_pred),
        mean_absolute_error(test, arima_pred),
        mean_absolute_error(test, hw_pred),
        mean_absolute_error(test, xgb_pred),
    ],
    "sMAPE (%)": [
        smape(test, naive_pred),
        smape(test, ma_pred),
        smape(test, arima_pred),
        smape(test, hw_pred),
        smape(test, xgb_pred),
    ],
})

# Sort by RMSE ascending (best model first)
results = results.sort_values("RMSE").reset_index(drop=True)
results.index += 1   # Start ranking from 1

print("=== Model Performance — Mixed Workload (73-day test set) ===")
print()
print(results.round(4).to_string())
print()
best_model = results.iloc[0]["Model"]
print(f"Best model by RMSE: {best_model}")


---
## Step 17 — Visualise Model Performance

### 17a — RMSE Comparison Bar Chart


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colours = ["#d62728", "#ff7f0e", "#2ca02c", "#1f77b4", "#9467bd"]
bars = ax.bar(results["Model"], results["RMSE"], color=colours, edgecolor="black", linewidth=0.6)

# Annotate each bar with its exact value
for bar, val in zip(bars, results["RMSE"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f"{val:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_title("RMSE Comparison — All Models (Mixed Workload)", fontsize=13, fontweight="bold")
ax.set_xlabel("Model")
ax.set_ylabel("RMSE (GB)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("graph_5.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved as graph_5.png")


### 17b — Actual vs Predicted Demand (Test Window)

This plot directly shows how each model tracks the real storage demand  
over the 73-day test period.


In [ ]:
plt.figure(figsize=(13, 5))
plt.plot(test.values,  label="Actual",       color="black",       linewidth=2.0)
plt.plot(arima_pred,   label="ARIMA",        color="steelblue",   linewidth=1.5, linestyle="--")
plt.plot(hw_pred,      label="Holt-Winters", color="darkorange",  linewidth=1.5, linestyle="--")
plt.plot(xgb_pred,     label="XGBoost",      color="forestgreen", linewidth=1.5, linestyle="--")
plt.title("Actual vs. Predicted Storage Demand — Mixed Workload (Test Period)", fontsize=13, fontweight="bold")
plt.xlabel("Day (test window)")
plt.ylabel("Storage Demand (GB)")
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("graph_6.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved as graph_6.png")


### 17c — XGBoost Residual (Error) Distribution

Plotting the distribution of residuals (actual − predicted) reveals  
whether errors are randomly distributed (desired) or systematically biased.  
A near-Gaussian distribution centred at zero is a sign of a well-behaved model.


In [ ]:
residuals = test.values - xgb_pred

plt.figure(figsize=(8, 4))
plt.hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
plt.axvline(0, color="red", linewidth=1.5, linestyle="--", label="Zero error")
plt.title("XGBoost Forecast Error Distribution (Residuals)", fontsize=12, fontweight="bold")
plt.xlabel("Residual (Actual − Predicted) in GB")
plt.ylabel("Frequency")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("graph_7.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Mean residual : {residuals.mean():.4f} GB  (closer to 0 = less bias)")
print(f"Std  residual : {residuals.std():.4f} GB")


---
## Step 18 — Multi-Horizon Evaluation (sMAPE at 7 / 14 / 28 / 45 / 90 days)

A single accuracy number hides important information: does the model  
stay accurate as you ask it to look further ahead?

We compute sMAPE at five forecasting horizons to answer this question.  
This is critical because different operational decisions need different lead times:

| Horizon | Use case |
|---|---|
| 7 days | Reactive auto-scaling decisions |
| 14 days | Weekly capacity reviews |
| 28 days | Monthly planning cycles |
| 45 days | Mid-quarter resource adjustments |
| 90 days | Quarterly infrastructure procurement |


In [ ]:
horizons = [7, 14, 28, 45, 90]

def evaluate_horizon(actual, pred, horizons):
    """
    Compute sMAPE at each specified forecasting horizon.
    
    Parameters
    ----------
    actual   : array-like  Ground-truth test values
    pred     : array-like  Model forecast values
    horizons : list[int]   Horizon lengths in days
    
    Returns
    -------
    dict  {horizon: sMAPE_value}
    """
    return {h: smape(actual[:h], pred[:h]) for h in horizons}

# Compute horizon-wise sMAPE for each model
arima_h = evaluate_horizon(test.values, arima_pred, horizons)
hw_h    = evaluate_horizon(test.values, hw_pred,    horizons)
xgb_h   = evaluate_horizon(test.values, xgb_pred,   horizons)

horizon_df = pd.DataFrame({
    "Horizon (days)" : horizons,
    "ARIMA sMAPE (%)" : [arima_h[h] for h in horizons],
    "Holt-Winters sMAPE (%)" : [hw_h[h]  for h in horizons],
    "XGBoost sMAPE (%)"      : [xgb_h[h] for h in horizons],
})

print("=== Multi-Horizon sMAPE Comparison — Mixed Workload ===")
print()
print(horizon_df.round(3).to_string(index=False))


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(horizon_df["Horizon (days)"], horizon_df["ARIMA sMAPE (%)"],
         label="ARIMA",        marker="o", linewidth=1.8, color="steelblue")
plt.plot(horizon_df["Horizon (days)"], horizon_df["Holt-Winters sMAPE (%)"],
         label="Holt-Winters", marker="s", linewidth=1.8, color="darkorange")
plt.plot(horizon_df["Horizon (days)"], horizon_df["XGBoost sMAPE (%)"],
         label="XGBoost",      marker="^", linewidth=1.8, color="forestgreen")
plt.title("Forecast Error (sMAPE) vs. Forecasting Horizon", fontsize=13, fontweight="bold")
plt.xlabel("Forecasting Horizon (days)")
plt.ylabel("sMAPE (%)")
plt.xticks(horizons)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("graph_8.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved as graph_8.png")


---
## Step 19 — Dynamic Cost Optimization Policy

Once we have a reliable forecast, we can use it to make **smarter provisioning decisions**  
instead of always provisioning at full predicted capacity.

### The Tiered Policy

We apply a three-tier storage allocation policy based on the XGBoost forecast:

| Predicted Demand | Action | Allocation |
|---|---|---|
| < 120 GB | Light consolidation | Provision at 95% of prediction |
| 120 – 160 GB | Moderate consolidation | Provision at 85% of prediction |
| > 160 GB | Aggressive deduplication | Provision at 70% of prediction |

### Cost Model
We use a simple flat pricing model: **$0.02 USD per GB per day**  
(representative of standard cloud object storage pricing).

> This policy saves money by avoiding over-provisioning, while the  
> safety margins ensure SLA compliance is not materially affected.


In [ ]:
price_per_gb = 0.02   # USD per GB per day

# Cost based on raw XGBoost forecast (no policy)
predicted_cost = xgb_pred * price_per_gb

# Cost based on actual demand
actual_cost = test.values * price_per_gb

print(f"Pricing assumption : ${price_per_gb}/GB/day")
print(f"Test window        : {len(test)} days")
print()
print(f"Avg. forecast cost : ${predicted_cost.mean():.4f}/day")
print(f"Avg. actual cost   : ${actual_cost.mean():.4f}/day")


In [ ]:
# Apply the tiered cost optimization policy
optimized_storage = []
tier_counts = {1: 0, 2: 0, 3: 0}   # Track how often each tier fires

for val in xgb_pred:
    if val < 120:
        optimized_storage.append(val * 0.95)   # Tier 1: light consolidation
        tier_counts[1] += 1
    elif val < 160:
        optimized_storage.append(val * 0.85)   # Tier 2: moderate consolidation
        tier_counts[2] += 1
    else:
        optimized_storage.append(val * 0.70)   # Tier 3: aggressive deduplication
        tier_counts[3] += 1

optimized_storage = np.array(optimized_storage)
optimized_cost    = optimized_storage * price_per_gb

print("=== Tiered Policy Application ===")
print(f"Tier 1 (< 120 GB, 5% cut)  : {tier_counts[1]} days")
print(f"Tier 2 (120-160 GB, 15% cut): {tier_counts[2]} days")
print(f"Tier 3 (> 160 GB, 30% cut)  : {tier_counts[3]} days")


In [ ]:
# Summarise cost savings
daily_saving = predicted_cost.mean() - optimized_cost.mean()
pct_saving   = (daily_saving / predicted_cost.mean()) * 100
annual_proj  = daily_saving * 365

policy_summary = pd.DataFrame({
    "Metric": [
        "Avg. Predicted Cost ($/day)",
        "Avg. Optimised Cost ($/day)",
        "Daily Saving ($/day)",
        "Saving (%)",
        "Projected Annual Saving ($)"
    ],
    "Value": [
        f"${predicted_cost.mean():.4f}",
        f"${optimized_cost.mean():.4f}",
        f"${daily_saving:.4f}",
        f"{pct_saving:.2f}%",
        f"${annual_proj:.2f}"
    ]
})

print("=== Cost Optimization Summary ===")
print()
print(policy_summary.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: predicted vs actual cost
axes[0].plot(actual_cost,    label="Actual Cost",    color="black",       linewidth=1.5)
axes[0].plot(predicted_cost, label="Predicted Cost", color="steelblue",   linewidth=1.5, linestyle="--")
axes[0].set_title("Predicted vs. Actual Storage Cost", fontweight="bold")
axes[0].set_xlabel("Day (test window)")
axes[0].set_ylabel("Cost (USD/day)")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Right: original vs optimised cost
axes[1].plot(predicted_cost, label="Original Provisioning",  color="steelblue",  linewidth=1.5)
axes[1].plot(optimized_cost, label="Optimised (Policy)",     color="forestgreen", linewidth=1.5, linestyle="--")
axes[1].fill_between(range(len(predicted_cost)), optimized_cost, predicted_cost,
                     alpha=0.15, color="forestgreen", label="Savings")
axes[1].set_title("Cost Reduction via Dynamic Optimization Policy", fontweight="bold")
axes[1].set_xlabel("Day (test window)")
axes[1].set_ylabel("Cost (USD/day)")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("graph_10_11.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figures saved as graph_10_11.png")


---
## Step 20 — Cross-Workload Stability Analysis

To understand whether model performance on Mixed generalises to other workload types,  
we now evaluate all three models on **all four workloads** using the same 80/20 split.

This answers the key question: *Is XGBoost always best, or does the winner change  
depending on the workload type?*

**Expected findings:**
- ARIMA should win on **Steady** (clean linear trend matches AR assumptions)  
- Holt-Winters should win on **Seasonal** (explicit seasonal component)  
- XGBoost should win on **Bursty** and **Mixed** (non-linear flexibility)


In [ ]:
# Dictionary of all four workload series
workloads = {
    "Steady"  : df["Steady"],
    "Seasonal": df["Seasonal"],
    "Bursty"  : df["Bursty"],
    "Mixed"   : df["Mixed"],
}

stability_results = []

for workload_name, series_w in workloads.items():
    # 80/20 split
    n          = len(series_w)
    split      = int(n * 0.8)
    train_w    = series_w[:split]
    test_w     = series_w[split:]

    # ── ARIMA ────────────────────────────────────────────────────────────
    arima_w    = ARIMA(train_w, order=(2, 1, 2)).fit()
    arima_pw   = arima_w.forecast(len(test_w))

    # ── Holt-Winters ─────────────────────────────────────────────────────
    hw_w       = ExponentialSmoothing(train_w, trend="add", seasonal=None).fit()
    hw_pw      = hw_w.forecast(len(test_w))

    # ── XGBoost ──────────────────────────────────────────────────────────
    Xt         = np.arange(len(train_w)).reshape(-1, 1)
    yt         = train_w.values
    xgb_w      = XGBRegressor(n_estimators=100, learning_rate=0.1,
                               max_depth=6, random_state=42, verbosity=0)
    xgb_w.fit(Xt, yt)
    Xf         = np.arange(len(train_w), len(train_w) + len(test_w)).reshape(-1, 1)
    xgb_pw     = xgb_w.predict(Xf)

    # Store RMSE for each model
    for model_name, preds in [("ARIMA", arima_pw), ("Holt-Winters", hw_pw), ("XGBoost", xgb_pw)]:
        rmse = np.sqrt(mean_squared_error(test_w.values, preds))
        stability_results.append({
            "Workload": workload_name,
            "Model"   : model_name,
            "RMSE (GB)": round(rmse, 3)
        })
    print(f"  {workload_name} — done")

stability_df = pd.DataFrame(stability_results)
stability_pivot = stability_df.pivot(index="Workload", columns="Model", values="RMSE (GB)")

print()
print("=== Cross-Workload RMSE Comparison (GB) ===")
print()
print(stability_pivot.to_string())
print()
# Identify winner for each workload
print("=== Best Model per Workload ===")
for wl in stability_pivot.index:
    best = stability_pivot.loc[wl].idxmin()
    val  = stability_pivot.loc[wl].min()
    print(f"  {wl:10s}: {best}  (RMSE = {val:.3f} GB)")


In [ ]:
# Grouped bar chart — RMSE per workload per model
workload_labels = stability_pivot.index.tolist()
model_names     = stability_pivot.columns.tolist()
x = np.arange(len(workload_labels))
width = 0.25
colours_bar = ["steelblue", "darkorange", "forestgreen"]

fig, ax = plt.subplots(figsize=(11, 5))
for i, (model, colour) in enumerate(zip(model_names, colours_bar)):
    bars = ax.bar(x + i * width, stability_pivot[model], width,
                  label=model, color=colour, edgecolor="black", linewidth=0.5)

ax.set_title("Cross-Workload RMSE Comparison — All Models", fontsize=13, fontweight="bold")
ax.set_xlabel("Workload Type")
ax.set_ylabel("RMSE (GB)")
ax.set_xticks(x + width)
ax.set_xticklabels(workload_labels)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("graph_9.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved as graph_9.png")


---
## Step 21 — Master Comparative Results Table

This table consolidates all key metrics for the three main models  
(ARIMA, Holt-Winters, XGBoost) on the Mixed workload.  
It is the single reference table corresponding to **Table 4.7** in the thesis.


In [ ]:
# Build master table from the already-computed results DataFrame
main_models = ["ARIMA", "Holt-Winters", "XGBoost"]
master = results[results["Model"].isin(main_models)].copy()
master = master.reset_index(drop=True)
master.index += 1

# Add 90-day horizon sMAPE column
master["sMAPE@90d (%)"] = [
    round(arima_h[90], 3),
    round(hw_h[90],    3),
    round(xgb_h[90],   3),
]

print("=== Master Comparative Results — Mixed Workload ===")
print()
print(master[["Model", "RMSE", "MAE", "sMAPE (%)", "sMAPE@90d (%)"]].round(3).to_string())
print()
print("Table 4.7 from thesis — reproduced from code.")


In [ ]:
# XGBoost feature importance (single feature: time index)
importance = xgb_model.feature_importances_

plt.figure(figsize=(5, 3))
plt.bar(["Time Index"], importance, color="steelblue", edgecolor="black")
plt.title("XGBoost Feature Importance", fontweight="bold")
plt.ylabel("Importance Score")
plt.ylim(0, 1.1)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("graph_13.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Feature importance of 'Time Index': {importance[0]:.4f}")
print("(= 1.0 because time index is the only input feature in this baseline)")


---
## Step 22 — Final Cost Summary

A clean, publication-ready summary of the cost optimization results,  
corresponding to **Table 4.5** in the thesis.


In [ ]:
cost_summary = pd.DataFrame({
    "Metric": [
        "Avg. Predicted Cost (USD/day)",
        "Avg. Optimised Cost (USD/day)",
        "Daily Saving (USD/day)",
        "Saving (%)",
        "Projected Annual Saving (USD)"
    ],
    "Value": [
        round(predicted_cost.mean(), 4),
        round(optimized_cost.mean(), 4),
        round(daily_saving, 4),
        round(pct_saving, 2),
        round(annual_proj, 2)
    ]
})

print("=== Cost Optimization Summary (Table 4.5) ===")
print()
print(cost_summary.to_string(index=False))


---
## Summary and How to Extend This Work

### What this notebook demonstrated

| Step | What was done |
|---|---|
| 1–5 | Installed libraries, set seed, defined time axis and parameters |
| 6–8 | Generated four synthetic workloads and explored their statistics |
| 9 | Visualised all workloads individually and via rolling statistics |
| 10 | Split data into training (80%) and test (20%) sets |
| 11 | Built naive baselines |
| 12 | Fitted ARIMA(2,1,2) using MLE |
| 13 | Fitted Holt-Winters with additive trend |
| 14 | Fitted XGBoost with time-index feature |
| 15–17 | Computed RMSE, MAE, sMAPE; plotted results |
| 18 | Evaluated all models at 5 forecasting horizons |
| 19 | Designed and evaluated a 3-tier cost optimization policy |
| 20 | Cross-workload stability analysis across all 4 workload types |
| 21–22 | Produced master results and cost summary tables |

---

### How to extend this notebook

**Add a new forecasting model:**
```python
from sklearn.linear_model import Ridge
ridge_model = Ridge().fit(X_train, y_train)
ridge_pred  = ridge_model.predict(X_test)
```

**Add lag features to XGBoost:**
```python
def make_lag_features(series, lags=[1, 7, 14]):
    df_feat = pd.DataFrame({'y': series})
    for lag in lags:
        df_feat[f'lag_{lag}'] = df_feat['y'].shift(lag)
    return df_feat.dropna()
```

**Test on a real dataset:**
```python
# Replace the synthetic workloads with a real CSV
df_real = pd.read_csv("your_cloud_trace.csv", parse_dates=["date"], index_col="date")
series  = df_real["storage_gb"]
```

**Run 10-fold time-series cross-validation:**
```python
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=10)
for fold, (train_idx, test_idx) in enumerate(tscv.split(series)):
    train_fold = series.iloc[train_idx]
    test_fold  = series.iloc[test_idx]
    # fit and evaluate here
```

---

### Citation

If you use this code in your research, please cite:

```
Panigrahi, J. (2026). Cloud Storage Demand Forecasting, Cost Optimization, 
and Policy Simulation Using Time-Series Machine Learning Models.
M. Tech Thesis, IMIT Cuttack, Biju Patnaik University of Technology.
GitHub: https://github.com/Jagannath_Panigrahi/cloud-storage-forecasting
```

---

*Last updated: July 2026 | License: IMIT*
